[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alanturin-g/Computational_Neuroscience_UNITO_2026/blob/main/Orsenigo_1_brain_modeling_neuron_to_whole_brain_solutions.ipynb)

# Lesson 1 — Brain modeling: from the neuron to the whole brain (solutions)

**Computational Neuroscience 2026/27 · Davide Orsenigo**

**Wednesday 4 November 2026, room G**

This notebook follows Chapter 3 (*Principles*) of

> G. Deco and M. L. Kringelbach, *Whole-brain modelling: Cartography of the dynamics of mind*, Oxford University Press (2026), open access (CC BY-NC-ND 4.0), doi:[10.1093/oso/9780198991250.001.0001](https://doi.org/10.1093/oso/9780198991250.001.0001).

Equation numbers (3.x) refer to the book. In six parts we go from one neuron to a model of the whole cortex:
68 regions, each with a dynamics of its own, coupled through the structural connectivity (SC) and compared with
the functional connectivity (FC) measured at rest. The last part brings back the information theory of Lesson 0.
The slides of the lesson follow the same path: their Part I is Parts 1–2 here, Part II is Part 3, Part III is
Part 4, Part IV is Parts 5–6. The labs A, B, C and D use the parts below.

| Part | Lab | Book | Content |
|---|---|---|---|
| 1 | A | pp. 40–41 | One neuron and its rate curve |
| 2 | A | eq. 3.1–3.6 | One region: the dynamic mean-field (DMF) model and feedback inhibition control (FIC) |
| 3 | B | eq. 3.12 | The Hopf node: the normal form of an oscillation near its onset, driven by noise |
| 4 | C | eq. 3.13–3.14 | The whole-brain Hopf model: 68 nodes coupled through the SC |
| 5 | D | eq. 3.15–3.19 | FC without simulation: the linear theory |
| 6 | D | Lesson 0 | Information in the model: mutual information and O-information |

Everything runs on the default Colab runtime (numpy, scipy, matplotlib): no installation and no Google Drive
needed. Every cell runs in a few seconds.

Solutions to the seven exercises in `Orsenigo_1_brain_modeling_neuron_to_whole_brain`, with answers to the
questions. Try on your own first, then compare.

---
# Setup

The data are group averages from the Human Connectome Project (HCP), distributed with the ENIGMA Toolbox
(Larivière et al., *Nat Methods* 18:698, 2021). The cortex, the outer layer of the brain, is divided into 68
regions with a standard map, the Desikan–Killiany parcellation: 34 in each hemisphere (the two halves of the
brain). Two matrices give one number for each pair of regions $i$, $j$:

- **SC** (structural connectivity): the strength of the nerve fibres between the two regions, in arbitrary units.
  Diffusion MRI (magnetic resonance imaging) follows water, which moves most easily along the fibres, and
  tractography rebuilds the fibres (*streamlines*) from it. The SC is symmetric: tractography cannot tell a
  fibre's direction.
- **FC** (functional connectivity): the correlation between the BOLD signals of the two regions. Functional MRI
  (fMRI) records the BOLD signal (blood-oxygen-level dependent), which follows neural activity through the blood
  flow, a few seconds late. It was recorded at rest (lying in the scanner, with no task), and the FC is averaged
  over the participants.

In [ ]:
import hashlib
import os
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_continuous_lyapunov

# Figure style: minimal axes, Arial-like font, palette and colormaps of figkit (D. Orsenigo)
from matplotlib.colors import LinearSegmentedColormap
PALETTE = ["#3B6E8F", "#E07A5F", "#81B29A", "#E0A458", "#8D6A9F", "#C65B6B", "#5B7080", "#3D3D3D"]
FC_CMAP = LinearSegmentedColormap.from_list(   # diverging petrol-brick, white at 0
    "fk_div", ["#2C5F86", "#6FA0BE", "#BFD3DF", "#F3F3F1", "#EFC0B2", "#D77C6A", "#B23A48"])
SC_CMAP = LinearSegmentedColormap.from_list(   # sequential cream-plum
    "fk_warm", ["#FBEEE6", "#F6C19A", "#EE8A5C", "#D8503F", "#A4255B", "#5E1246"])
plt.rcParams.update({
    "figure.dpi": 110, "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
    "font.size": 10, "axes.titlesize": 10.5, "axes.labelsize": 10, "legend.fontsize": 9,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "mathtext.fontset": "dejavusans",
    "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.8, "axes.grid": False,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.prop_cycle": plt.cycler(color=PALETTE),
    "xtick.direction": "out", "ytick.direction": "out", "xtick.major.size": 3, "ytick.major.size": 3,
    "xtick.major.width": 0.8, "ytick.major.width": 0.8, "lines.linewidth": 1.4, "lines.markersize": 4.5,
    "legend.frameon": False, "image.cmap": "magma"})

# Download the three ENIGMA files (about 150 kB) into the working directory, if they are not there yet.
# The address names one commit of the ENIGMA repository, and the SHA-256 of each file is checked.
ENIGMA = ("https://raw.githubusercontent.com/MICA-MNI/ENIGMA/cf6917af2afcc642d70402fc96ba8c30c711b1a5/"
          "enigmatoolbox/datasets/matrices/hcp_connectivity/")
SHA256 = {"strucMatrix_ctx.csv": "e57b94ec8de0b30f128899413b02a09538a2911633db41fff00ff4715d5841b5",
          "funcMatrix_ctx.csv": "8c4fbbe79dc851ac51bff40bf1bdfd07ce839f819d74519ad7518a731e07b934",
          "strucLabels_ctx.csv": "47a5301cb4f956782fa99c6c0b110fc4717c4bb626a65a4d04890ed72a745a35"}
for file_name, digest in SHA256.items():
    if not os.path.exists(file_name):
        urllib.request.urlretrieve(ENIGMA + file_name, file_name)
    if hashlib.sha256(open(file_name, "rb").read()).hexdigest() != digest:
        print(f"Warning: {file_name} is not the file this notebook was written for")
print("data ready")

---
# Part 1 — One neuron and its rate curve

The first model of the book is built from networks of spiking neurons (p. 41), so we start from one neuron: the
leaky integrate-and-fire (LIF) neuron. Its membrane potential $V$ is the voltage across the cell membrane, in mV,
measured from its resting value. The membrane is a capacitor $c$ that leaks charge through a resistance $R$: with
an input current $I_{in}$, $V$ relaxes exponentially to $\mu = R\, I_{in}$ with the time constant $\tau_m = Rc = 20$ ms. The input,
thousands of irregular spikes from other neurons, also fluctuates, and the noise $\xi(t)$ stands for these
fluctuations:

$$\tau_m \frac{dV}{dt} = -(V - \mu) + \xi(t)$$

When $V$ reaches the threshold $\theta = 20$ mV the neuron fires a spike, a brief electrical pulse: $V$ is reset
to $V_r = 10$ mV and stays there for a pause of 2 ms (`t_ref` in the code). The noise is white,
$\langle \xi(t)\,\xi(t') \rangle = \sigma^2 \tau_m\, \delta(t - t')$: without threshold, $V$ would fluctuate around
$\mu$ with standard deviation $\sigma/\sqrt{2}$. In the code, each time step `dt` adds $\sigma\sqrt{dt/\tau_m}$ times
a standard normal number.

Many similar neurons, each with its own noise, fire on average at a rate (spikes per second) that depends on
$\mu$ and $\sigma$: the **rate curve**, which neuroscientists call the *transfer function*. The next cell
simulates 200 independent neurons for each value of $\mu$, without noise and with $\sigma = 4$ mV. Part 2 uses
a smooth formula for this curve.

In [ ]:
def lif_rates(mus, sigma, n_neurons=200, T=1.0, dt=1e-4, tau_m=0.02, theta=20.0, V_reset=10.0,
              t_ref=0.002, seed=0):
    """Firing rate (Hz) of n_neurons independent LIF neurons for each mean input in mus (mV)."""
    rng = np.random.default_rng(seed)
    mu = np.asarray(mus, dtype=float)[:, None]
    V = np.full((len(mu), n_neurons), V_reset)
    refractory = np.zeros_like(V)            # time left in the refractory period (s)
    n_spikes = np.zeros(len(mu))
    for _ in range(int(T / dt)):
        dV = dt / tau_m * (mu - V) + sigma * np.sqrt(dt / tau_m) * rng.standard_normal(V.shape)
        V = np.where(refractory <= 0, V + dV, V)
        refractory -= dt
        spike = V >= theta
        n_spikes += spike.sum(axis=1)
        V[spike] = V_reset
        refractory[spike] = t_ref
    return n_spikes / (n_neurons * T)


mus = np.arange(10, 31, 2.0)
plt.figure(figsize=(5, 3.5))
for sigma in (0.0, 4.0):
    plt.plot(mus, lif_rates(mus, sigma), "o-", label=f"σ = {sigma:.0f} mV")
plt.axvline(20, color="#7F7F7F", ls=":", lw=1, label="threshold θ")
plt.xlabel("mean input μ (mV)")
plt.ylabel("firing rate (Hz)")
plt.legend()
plt.show()

**Question 1.** Without noise the neuron is silent for $\mu \le \theta$. Why does it fire below threshold when
$\sigma > 0$, and what does this do to the shape of the rate curve?

**Answer.** Fluctuations carry $V$ across $\theta$ even when its mean is below threshold, so the rate is positive below
$\theta$. Without noise the rate rises from zero at $\mu = \theta$ with infinite slope,
$r = [t_{ref} + \tau_m \ln\frac{\mu - V_r}{\mu - \theta}]^{-1}$, with $t_{ref} = 2$ ms the pause after each spike.
Noise removes this singularity: below threshold the rate falls roughly as $\exp[-(\theta - \mu)^2/\sigma^2]$, and above it grows almost linearly:
the shape described by $H$ in eq. 3.3–3.4 (Part 2).

---
# Part 2 — One region: the dynamic mean-field model (eq. 3.1–3.6)

A brain region holds hundreds of millions of neurons of two kinds: excitatory (E) neurons raise the input current
of the neurons they reach, inhibitory (I) ones lower it. In the dynamic mean-field (DMF) model of Deco et al.
(2013, 2014), based on Wong and Wang (2006), each region has two populations, E and I (the book calls them
*pools*), each described by its firing rate $r$, in spikes per neuron per second.

A spike reaching a synapse, the contact between two neurons, opens receptors on the next neuron, which let a
current through. NMDA receptors (excitatory) close in about 100 ms, GABA receptors (inhibitory) in about 10 ms.
The DMF takes the rate to follow the input current instantly, $r = H(I)$: an adiabatic approximation, justified
because in a large population many neurons are near threshold. Its variables are the fractions $S$ of open
receptors, NMDA for the E population and GABA for the I population; only the NMDA fraction is slow.

Each population closes a loop: its open receptors set its input current $I$ (in nA), the current sets its rate
through the rate curve $H$, and the rate opens receptors. For one region, with the input from the other regions
(a term of eq. 3.1 with the global coupling $G$ of Part 4) replaced by a constant extra input current $I_{ext}$:

$$I^{(E)} = W_E I_0 + w_+ J_{NMDA} S^{(E)} - J\, S^{(I)} + I_{ext}, \qquad
  I^{(I)} = W_I I_0 + J_{NMDA} S^{(E)} - S^{(I)} \qquad (3.1\text{–}3.2)$$

$$r^{(E,I)} = H^{(E,I)}\big(I^{(E,I)}\big) =
  \frac{g_{E,I}\,\big(I^{(E,I)} - I^{(E,I)}_{thr}\big)}{1 - \exp\big[-d_{E,I}\, g_{E,I}\,\big(I^{(E,I)} - I^{(E,I)}_{thr}\big)\big]}
  \qquad (3.3\text{–}3.4)$$

$$\frac{dS^{(E)}}{dt} = -\frac{S^{(E)}}{\tau_{NMDA}} + \gamma\,\big(1 - S^{(E)}\big)\, r^{(E)} + \xi(t), \qquad
  \frac{dS^{(I)}}{dt} = -\frac{S^{(I)}}{\tau_{GABA}} + r^{(I)} + \xi'(t) \qquad (3.5\text{–}3.6)$$

Read the current of E term by term: a constant background input, $W_E I_0$; excitation from its own open
receptors, $w_+ J_{NMDA} S^{(E)}$; local inhibition, $-J\,S^{(I)}$, whose strength $J$ (in nA) is what feedback
inhibition control tunes (Exercise 2); and the extra input $I_{ext}$. The I population receives the background,
excitation from E, and its own inhibition, of strength 1 nA. In eq. 3.5 open receptors close by themselves in
$\tau_{NMDA} = 100$ ms, and spikes open the closed ones at a rate set by $\gamma = 0.641$, hence $1 - S$: when all
are open, no more can open. GABA receptors (eq. 3.6) close in 10 ms and do not saturate. The noises $\xi$ and
$\xi'$, independent and of size $\sigma = 0.01$, are there because a population has a finite number of neurons.
Currents are in nA, rates in Hz, times in s. The parameters (p. 42) are in the next cell.

In [ ]:
DMF = dict(W_E=1.0, W_I=0.7, I_0=0.382, w_plus=1.4, J_NMDA=0.15,    # I_0 and J_NMDA in nA
           g_E=310.0, g_I=615.0, I_thr_E=0.403, I_thr_I=0.288,     # gains (Hz per nA); where H bends (nA)
           d_E=0.16, d_I=0.087,                                    # sharpness of the bend (s)
           tau_NMDA=0.1, tau_GABA=0.01, gamma=0.641, sigma=0.01)   # time constants (s); size of the noise on S

## Exercise 1: The function H

$H$ is the smooth rate curve of Part 1, now as a function of the input current $I$ (in nA) instead of $\mu$. Its
formula, eq. 3.3–3.4, is from Abbott and Chance (2005). For each population it has three parameters: the gain $g$
(how fast the rate grows, in Hz per nA), the current $I_{thr}$ where the curve bends, and $d$, which sets how
sharp the bend is. Write `H(I, g, I_thr, d)` for a number or an array `I`, then run the next cell to plot
$H^{(E)}$ and $H^{(I)}$. Exactly at $I = I_{thr}$ the formula is 0/0: there $H$ must return its limit, $1/d$
(`np.where` can pick it; `np.expm1(z)` computes $e^z - 1$ accurately for small $z$).

*Expected:* both curves rise smoothly, pass through $H = 1/d$ at $I = I_{thr}$ (6.25 Hz for E, 11.5 Hz for I,
printed by the next cell), then grow almost linearly with slope equal to the gain: 310 Hz/nA for E and
615 Hz/nA for I.

**Solution**

In [ ]:
def H(I, g, I_thr, d):
    """Rate curve of eq. 3.3-3.4: input current I (nA) -> firing rate (Hz)."""
    x = g * (np.asarray(I, dtype=float) - I_thr)
    with np.errstate(invalid="ignore"):          # 0/0 at I = I_thr, replaced below by the limit 1/d
        r = x / -np.expm1(-d * x)                # expm1(z) = e^z - 1, accurate also for small z
    return np.where(x == 0, 1 / d, r)[()]        # [()]: a number when I is a number

In [ ]:
I = np.linspace(0.15, 0.65, 400)
plt.figure(figsize=(5, 3.5))
plt.plot(I, H(I, DMF["g_E"], DMF["I_thr_E"], DMF["d_E"]), label="E population")
plt.plot(I, H(I, DMF["g_I"], DMF["I_thr_I"], DMF["d_I"]), label="I population")
plt.xlabel("input current I (nA)")
plt.ylabel("firing rate r (Hz)")
plt.legend()
plt.show()
print(f"at the threshold: H = {H(DMF['I_thr_E'], DMF['g_E'], DMF['I_thr_E'], DMF['d_E']):.2f} Hz for E, "
      f"{H(DMF['I_thr_I'], DMF['g_I'], DMF['I_thr_I'], DMF['d_I']):.2f} Hz for I")

The next cell follows one region on its own ($I_{ext} = 0$, $J = 1$ nA) for 3 s. It integrates eq. 3.1–3.6 in
small time steps (the Euler–Maruyama method, the simplest one for equations with noise) and plots the rate of the
E population. Wong and Wang (2006) chose the parameters so that a region on its own fires at a low rate, as
cortical neurons do at rest.

In [ ]:
def dmf_currents(S_E, S_I, J, I_ext, p=DMF):
    I_E = p["W_E"] * p["I_0"] + p["w_plus"] * p["J_NMDA"] * S_E - J * S_I + I_ext
    I_I = p["W_I"] * p["I_0"] + p["J_NMDA"] * S_E - S_I
    return I_E, I_I


def dmf_node(J=1.0, I_ext=0.0, T=3.0, dt=1e-4, seed=0, p=DMF):
    """One DMF region (eq. 3.1-3.6). Returns the time (s) and the rate r_E (Hz) of the E population, every 1 ms."""
    rng = np.random.default_rng(seed)
    S_E, S_I = 0.1, 0.1
    t, r_E = [], []
    for k in range(int(T / dt)):
        I_E, I_I = dmf_currents(S_E, S_I, J, I_ext, p)
        rE = H(I_E, p["g_E"], p["I_thr_E"], p["d_E"])
        rI = H(I_I, p["g_I"], p["I_thr_I"], p["d_I"])
        noise = p["sigma"] * np.sqrt(dt) * rng.standard_normal(2)
        S_E += dt * (-S_E / p["tau_NMDA"] + (1 - S_E) * p["gamma"] * rE) + noise[0]
        S_I += dt * (-S_I / p["tau_GABA"] + rI) + noise[1]
        S_E, S_I = min(max(S_E, 0.0), 1.0), max(S_I, 0.0)
        if k % 10 == 0:
            t.append(k * dt)
            r_E.append(rE)
    return np.array(t), np.array(r_E)


t, r_E = dmf_node()
plt.figure(figsize=(7, 2.5))
plt.plot(t, r_E, lw=0.8)
plt.xlabel("time (s)")
plt.ylabel("E rate r (Hz)")
plt.show()
print(f"mean rate of the E population after the first 0.5 s: {r_E[t > 0.5].mean():.2f} Hz")

On its own the region settles at a fixed point, where the E population fires at about 3.1 Hz. There its input
current, $I^{(E)} \approx 0.378$ nA, is below the bend of $H$ at $I_{thr} = 0.403$ nA: single neurons fire only
thanks to their input fluctuations, which $H$ already includes (Question 1). The eigenvalues of this fixed point
are real, $-5.95$ and $-232$ s$^{-1}$ (time constants 168 and 4 ms): a perturbation decays without oscillating.
This stable fixed point is the *asynchronous* state of the book.

## Exercise 2: Feedback inhibition control

In the whole-brain version of the model each region also receives excitation from the others, through the SC;
the hubs, the most connected regions, receive the most. A constant extra input $I_{ext} = 0.02$ nA raises the rate of the E population from about 3 Hz to about 10 Hz (next cell). Following Deco et
al. (2014), the book adds **feedback inhibition control** (FIC, p. 42), a model of inhibitory plasticity: a
feedback rule changes the strength $J$ of the local inhibition of each region until the region fires at 3 Hz, a
typical rate of cortical neurons at rest. With FIC the model fits the measured FC better.

Implement FIC for one region with the rule

$$J \leftarrow J + \eta\,\big(r - 3\ \text{Hz}\big), \qquad \eta = 0.02,$$

where $r$ is the rate of the E population and $\eta$ (in nA per Hz) sets the size of each step. The function
`dmf_rate(J, I_ext)` below returns the steady rate of the E population, without noise.

*Expected:* after 40 steps, $J \approx 1.53$ nA and $r \approx 3.0$ Hz.

In [ ]:
def dmf_rate(J, I_ext, T=2.0, dt=1e-3, p=DMF):
    """Steady rate (Hz) of the E population of one DMF region, without noise, after integrating for T seconds."""
    S_E, S_I = 0.17, 0.04
    for _ in range(int(T / dt)):
        I_E, I_I = dmf_currents(S_E, S_I, J, I_ext, p)
        S_E += dt * (-S_E / p["tau_NMDA"] + (1 - S_E) * p["gamma"] * H(I_E, p["g_E"], p["I_thr_E"], p["d_E"]))
        S_I += dt * (-S_I / p["tau_GABA"] + H(I_I, p["g_I"], p["I_thr_I"], p["d_I"]))
    return H(dmf_currents(S_E, S_I, J, I_ext, p)[0], p["g_E"], p["I_thr_E"], p["d_E"])


print(f"J = 1 nA, no extra input:   r = {dmf_rate(1.0, 0.0):.2f} Hz")
print(f"J = 1 nA, I_ext = 0.02 nA:  r = {dmf_rate(1.0, 0.02):.2f} Hz")

**Solution**

In [ ]:
I_ext, eta = 0.02, 0.02
J = 1.0
for step in range(40):
    r = dmf_rate(J, I_ext)
    J = J + eta * (r - 3.0)

print(f"J = {J:.3f} nA,  r = {dmf_rate(J, I_ext):.2f} Hz")

In the whole-brain DMF the same rule is applied to all regions at once, with $I_{ext}$ replaced by the input from
the other regions, $G J_{NMDA} \sum_p C_{np} S^{(E)}_p$ for region $n$, with $C_{np}$ the SC between regions $n$
and $p$ and $G$ the global coupling of Part 4. The hubs end up with the strongest inhibition. The E rates then feed a model of the blood flow (the Balloon–Windkessel model, eq. 3.7–3.11; optional
part B at the end), which produces the BOLD signal.

---
# Part 3 — The Hopf node (eq. 3.12)

The DMF region has no rhythm of its own, has many parameters, and reaches the BOLD signal only through the blood.
The second model of the book skips the neurons and describes directly the slow rise and fall of the BOLD signal
of a region: after band-pass filtering, its power at rest peaks near 0.05 Hz, about once every 20 s.

An oscillation is described by its amplitude $\rho$ and its phase $\varphi$, the polar coordinates of a point
$(x, y)$. Without noise, the Hopf node is

$$\dot\rho = a\rho - \rho^3, \qquad \dot\varphi = \omega.$$

The sign of $a$ (in s$^{-1}$) decides growth or decay, $-\rho^3$ saturates large amplitudes, and
$\omega/2\pi = 0.05$ Hz is the frequency: one turn every 20 s. Near a supercritical Hopf bifurcation, a generic smooth
system, on its centre manifold and up to cubic order, reduces to this form after rescaling $\rho$, with
$\dot\varphi = \omega + b\rho^2$: its normal form (Strogatz, §8.2). The book sets $b = 0$. It is a model of the
BOLD signal, not of neurons.

- $a < 0$: $\dot\rho < 0$ for every $\rho > 0$, so the node spirals back to the origin, which is stable.
- $a > 0$: the origin is unstable, and $\dot\rho = 0$ at $\rho = \sqrt{a}$: a stable limit cycle.

The change of stability at $a = 0$ is a supercritical Hopf bifurcation. The book works just below it, at the
working point $a = -0.02$ s$^{-1}$, where the model fits the data best (p. 44): there a node does not oscillate by
itself, and noise and the other regions drive it. The book writes the node with the complex number
$z = x + iy = \rho\, e^{i\varphi}$ and adds noise:

$$\frac{dz}{dt} = (a + i\omega)\, z - |z|^2 z + \xi(t) \qquad (3.12\text{, one node})$$

$\xi(t)$ is white noise of size $\sigma = 0.02$ on $x$ and on $y$, and $x = \mathrm{Re}\, z$ is the model's BOLD
signal of the region. The book estimates the frequency region by region from the data (between 0.008 and 0.08 Hz); here
every region has 0.05 Hz. The function `simulate_hopf` below serves the whole notebook: without connections it
simulates nodes on their own.

In [ ]:
def simulate_hopf(C, G, a=-0.02, f=0.05, sigma=0.02, dt=0.1, TR=2.0, n_vol=600, burn=200.0, seed=0,
                  return_z=False, cubic=True):
    """Whole-brain Hopf model (eq. 3.13-3.14) with Euler-Maruyama; the rotation by omega*dt is applied exactly.
    C: (N, N) fibre strengths; G: one value or an array of values (simulated together); f: frequency in Hz;
    sigma: size of the noise; TR: time between two samples in s (one fMRI image every 2 s); n_vol: number of samples;
    burn: seconds simulated and discarded first, so that the start does not matter.
    cubic=False drops the cubic term -(x^2 + y^2): the linearised model.
    Returns x with shape (len(G), n_vol, N), or z = x + iy if return_z is True."""
    C = np.asarray(C, dtype=float)
    N = C.shape[0]
    G = np.atleast_1d(np.asarray(G, dtype=float))[:, None]          # (B, 1): one row per value of G
    a = np.asarray(a, dtype=float)                                  # a number, (N,) or (B, 1)
    B = max(len(G), a.shape[0] if a.ndim == 2 else 1)               # number of simulations run together
    s = C.sum(axis=1)                                               # strength of each region: sum of its fibres
    w = 2 * np.pi * f
    cw, sw = np.cos(w * dt), np.sin(w * dt)
    rng = np.random.default_rng(seed)
    x, y = 0.1 * rng.standard_normal((2, B, N))
    step, n_burn = int(round(TR / dt)), int(round(burn / dt))
    out = np.empty((B, n_vol, N), dtype=complex if return_z else float)
    for k in range(n_burn + n_vol * step):
        rho2 = x**2 + y**2 if cubic else 0.0                        # squared amplitude of each node
        xe = x + dt * ((a - rho2) * x + G * (x @ C.T - s * x))      # sum_i C_ji (x_i - x_j)
        ye = y + dt * ((a - rho2) * y + G * (y @ C.T - s * y))
        noise = sigma * np.sqrt(dt) * rng.standard_normal((2, B, N))
        x, y = cw * xe - sw * ye + noise[0], sw * xe + cw * ye + noise[1]
        if k >= n_burn and (k - n_burn) % step == 0:
            out[:, (k - n_burn) // step] = x + 1j * y if return_z else x
    return out

In [ ]:
# one node, a below and above the bifurcation: 2 rows, 20 minutes sampled every 0.5 s
z = simulate_hopf(np.zeros((1, 1)), 0.0, a=np.array([[-0.02], [0.02]]), TR=0.5, n_vol=2400,
                  return_z=True)[:, :, 0]
time_min = np.arange(z.shape[1]) * 0.5 / 60
fig, axes = plt.subplots(2, 1, figsize=(8, 4), sharex=True, sharey=True)
for ax, zi, a_val in zip(axes, z, (-0.02, 0.02)):
    ax.plot(time_min, zi.real, lw=0.7)
    ax.set_ylabel("x = Re z")
    ax.set_title(f"a = {a_val:+.2f}", fontsize=10)
axes[1].set_xlabel("time (min)")
plt.tight_layout()
plt.show()

## Exercise 3: The amplitudes

Compute the time average of the amplitude $\rho = |z|$, the distance of the point $(x, y)$ from the origin, for
the two simulations (`z` has one row per value of $a$). Compare it with the radius of the circle,
$\sqrt{0.02} \approx 0.141$.

*Expected:* for this simulation, 0.096 below the bifurcation and 0.137 above it. Long runs give 0.093 and 0.152:
with noise, the mean amplitude above the bifurcation exceeds the radius $\sqrt{a}$.

**Solution**

In [ ]:
amplitude = np.abs(z).mean(axis=1)

print(f"mean amplitude for a = -0.02: {amplitude[0]:.3f}")
print(f"mean amplitude for a = +0.02: {amplitude[1]:.3f}   (radius of the circle: {np.sqrt(0.02):.3f})")

**Question 2.** With this noise, a node just *below* the bifurcation fluctuates with a mean amplitude of about two
thirds of the radius $\sqrt{a}$. What does the working point $a = -0.02$ give us that $a = +0.02$ does not?

**Answer.** Below the bifurcation the fluctuations are driven by noise around a stable point, and the region responds to the
inputs from the others instead of following its own oscillation. In the network the coupling damps each node
(at $G = 8$ the variance of $x$ is about 10 times smaller than for the node on its own), so the fluctuations
become nearly linear and the FC can be computed without simulation (Part 5). For the node on its own the cubic term
$-\rho^3$ still matters: the linear theory would predict a mean amplitude
$\langle\rho\rangle = \sigma\sqrt{\pi/(4|a|)} \approx 0.125$, against the 0.096 you measured.

---
# Part 4 — The whole-brain Hopf model (eq. 3.13–3.14)

Now there is one Hopf node per region, $j = 1, \dots, 68$, with its two coordinates $x_j$ and $y_j$; $x_j$ is the
BOLD signal of region $j$. The regions are coupled through the fibre strengths $C_{ji}$:

$$\dot x_j = \underbrace{(a - x_j^2 - y_j^2)\,x_j - \omega\, y_j}_{\text{the Hopf node of Part 3}} +
  \underbrace{G \sum_i C_{ji}\,(x_i - x_j)}_{\text{coupling}} +
  \underbrace{\xi_j(t)}_{\text{noise}} \qquad (3.13)$$

$$\dot y_j = (a - x_j^2 - y_j^2)\,y_j + \omega\, x_j + G \sum_i C_{ji}\,(y_i - y_j) + \xi'_j(t) \qquad (3.14)$$

In $x$ and $y$, $x_j^2 + y_j^2 = \rho_j^2$ gives the cubic term and $\omega$ the rotation. The coupling acts only
on differences: summed over $i$ it is $-G\,(L x)_j$, with $L = \mathrm{diag}(s) - C$ the Laplacian of the SC and
$s_j = \sum_i C_{ji}$ the strength of region $j$. The eigenvalues $\lambda_m$ of $L$ are non-negative, and
$\lambda_0 = 0$ for the uniform pattern (all $x_j$ equal). The coupling leaves that pattern alone and damps every
other pattern $m$ at the extra rate $G\lambda_m$, so the linearised network is stable for every $G \ge 0$. The
noises are white and independent in each region and for $x$ and $y$,
$\langle \xi_j(t)\,\xi_j(t') \rangle = \sigma^2 \delta(t - t')$ with $\sigma = 0.02$.

- $a = -0.02$ s$^{-1}$ and $\omega/2\pi = 0.05$ Hz in every region.
- $C$: the SC, rescaled so that the largest entry is 0.2, following Deco et al. (2017). Only $G\,C$ enters the
  equations, so the rescaling just fixes the units of $G$.
- $G$: the global coupling, the only free parameter. $C$ gives the relative strengths of the connections, $G$
  their overall scale, which tractography cannot measure.

The model is compared with the data in two ways. The **fit** is the correlation between the FC of the model and
the measured FC over all 2278 pairs of regions: it compares patterns. The **mean FC** is the average of all the
correlations: it compares levels.

In [ ]:
SC_log = np.loadtxt("strucMatrix_ctx.csv", delimiter=",")    # ENIGMA stores the log of the streamline weights
SC = np.where(SC_log > 0, np.exp(SC_log), 0.0)               # linear weights
C = 0.2 * SC / SC.max()                                      # the fibre strengths of the model
# measured FC (emp = empirical). ENIGMA stores the correlations transformed by arctanh (the Fisher z),
# with negative values set to 0; tanh brings them back to correlations
FC_emp = np.tanh(np.loadtxt("funcMatrix_ctx.csv", delimiter=","))
np.fill_diagonal(FC_emp, 1.0)
labels = np.loadtxt("strucLabels_ctx.csv", dtype=str, delimiter=",")
N = len(C)


def upper(M):
    """Each pair of regions once: the upper triangle of a square matrix, diagonal excluded, as a vector."""
    return M[np.triu_indices(len(M), 1)]


def fc_fit(FC_model, FC_data):
    """The fit: correlation between two FC matrices over all pairs of regions."""
    return np.corrcoef(upper(FC_model), upper(FC_data))[0, 1]


fig, axes = plt.subplots(1, 2, figsize=(9, 4))
im = axes[0].imshow(np.log10(np.where(C > 0, C, np.nan)), vmin=-6, vmax=np.log10(0.2),
                    cmap=SC_CMAP.with_extremes(bad="white"))   # white: no fibre
axes[0].set_title("SC: C on a log scale (log10)")
plt.colorbar(im, ax=axes[0], shrink=0.8)
im = axes[1].imshow(FC_emp, vmin=-1, vmax=1, cmap=FC_CMAP)
axes[1].set_title("FC, data")
plt.colorbar(im, ax=axes[1], shrink=0.8)
for ax in axes:
    ax.set_xlabel("region")
    ax.grid(False)
plt.tight_layout()
plt.show()
print(f"N = {N} regions, {len(upper(C))} pairs; mean FC of the data = {upper(FC_emp).mean():.3f}; "
      f"the first region is {labels[0]}, region 35 is {labels[34]}")
print(f"fit of the SC alone (correlation between C and the measured FC over all pairs) = {fc_fit(C, FC_emp):.2f}")

The left hemisphere (34 regions) comes first, then the right one, in the same order. Most fibres stay within a
hemisphere, hence the two large blocks of the SC. The SC alone explains only part of the FC. The next cell
simulates 20 minutes of rest at $G = 8$, one value every 2 s as in a real fMRI scan (600 images, or *volumes*),
and compares the FC of the model with the data. $G = 8$ is a first guess; Exercise 4 scans $G$.

*Expected:* a fit of about 0.64. A simulation with other noise gives another fit, between 0.53 and 0.65 over ten
runs: Exercise 4, with another seed, gives 0.59 at $G = 8$.

In [ ]:
x = simulate_hopf(C, 8.0, seed=1)[0]          # shape (600 volumes, 68 regions)
FC_sim = np.corrcoef(x.T)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
im = axes[0].imshow(FC_sim, vmin=-1, vmax=1, cmap=FC_CMAP)
axes[0].grid(False)
axes[0].set_title("FC, model (G = 8)")
plt.colorbar(im, ax=axes[0], shrink=0.8)
axes[1].plot(upper(FC_emp), upper(FC_sim), ".", ms=2, alpha=0.4)
axes[1].plot([-0.2, 1], [-0.2, 1], "k:", lw=1)
axes[1].set_xlabel("FC, data")
axes[1].set_ylabel("FC, model")
plt.tight_layout()
plt.show()
print(f"fit (correlation over all pairs of regions) = {fc_fit(FC_sim, FC_emp):.3f}")
print(f"mean FC: model {upper(FC_sim).mean():.2f}, data {upper(FC_emp).mean():.2f}")

Each point of the scatter plot is a pair of regions, and the fit is the correlation of these points.

## Exercise 4: Scanning G

The cell below simulates seven values of $G$ at once, in one call. For each value, store the fit and the mean FC
of the model. The next cell plots both against $G$.

*Expected:* the fit grows with $G$ and approaches 0.65 for $G \ge 16$. The mean FC grows with $G$ and reaches the
value of the data, 0.29, near $G = 8$. It may not grow between two neighbouring values of $G$: a 20-minute
simulation has a random error.

**Solution**

In [ ]:
Gs = np.array([1, 2, 4, 8, 12, 16, 20])
X = simulate_hopf(C, Gs, seed=2)              # shape (len(Gs), 600, 68); a few seconds
fit, mean_fc = [], []
for x in X:
    FC_sim = np.corrcoef(x.T)
    fit.append(fc_fit(FC_sim, FC_emp))
    mean_fc.append(upper(FC_sim).mean())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(Gs, fit, "o-")
axes[0].set_xlabel("G")
axes[0].set_ylabel("fit")
axes[1].plot(Gs, mean_fc, "o-", label="model")
axes[1].axhline(upper(FC_emp).mean(), color="#1A1A1A", ls=":", lw=1, label="data")
axes[1].set_xlabel("G")
axes[1].set_ylabel("mean FC")
axes[1].legend()
plt.tight_layout()
plt.show()

---
# Part 5 — FC without simulation: the linear theory (eq. 3.15–3.19)

Simulations have a random error. Near the working point the linearised model gives the FC exactly. Each pattern of
activity $m$ is damped at the rate $|a| + G\lambda_m \ge |a|$ (Part 4), so the weakest damping is
$|a| = 0.02$ s$^{-1}$, that of the uniform pattern. The cubic term $-(x_j^2 + y_j^2)\,x_j = -\rho_j^2\, x_j$ acts, on
average, as an extra damping $2\langle\rho_j^2\rangle$, about 0.002 s$^{-1}$ at $G = 8$: a tenth of $|a|$. Neglecting
it, the model is **linear**. The FC of the linear model is exact. Simulations of the full model have a lower mean
FC, about 9% lower at $G = 8$, mostly because of the cubic term (end of this part), so they match the measured
mean FC at a slightly larger $G$, about 9.

Then the 136 coordinates $u = (x_1, \dots, x_{68}, y_1, \dots, y_{68})$ obey (eq. 3.16)

$$\dot u = J u + \xi(t), \qquad J = \begin{pmatrix} A & -B \\ B & A \end{pmatrix} \qquad (3.17)$$

$J$ is a $136 \times 136$ matrix built from $a$, $\omega$, $G$ and $C$ (the Jacobian of the model at the
origin). It has four blocks of size $68 \times 68$:

- $A = \mathrm{diag}(a - G s) + G\,C = a - G L$ acts within the $x$, and in the same way within the $y$. On the diagonal
  are $a$ and the damping by the neighbours, $-G s_j$; off the diagonal are the fibres, $G\,C_{ji}$.
- $B = \mathrm{diag}(\omega)$ links each $x_j$ with its own $y_j$: the rotation.

This $J$ has nothing to do with the strength $J$ of the inhibition in Part 2: the book uses the same letter for
both. The book also writes the opposite sign for the two $B$ blocks, which does not change the statistics of $x$.

The covariance $K = \langle u\, u^T \rangle$ evolves as $\dot K = J K + K J^T + Q$, where $Q$ is the intensity of
the noise, $\langle \xi(t)\,\xi^T(t') \rangle = Q\,\delta(t - t')$: $\sigma^2$ on the diagonal and 0 elsewhere,
because the noises are independent. At stationarity
$\dot K = 0$, the **Lyapunov equation**:

$$J K + K J^T + Q = 0 \qquad (3.19)$$

The FC is each covariance of the $x$ (the first 68 rows and columns of $K$) divided by the two standard
deviations. It is exact, with no random error and no simulation: 40 values of $G$ take about a second.

## Exercise 5: The matrix J

Complete the block $A$ of eq. 3.17. `np.diag(v)` builds a diagonal matrix from a vector `v`.

*Expected:* the largest real part of the eigenvalues of $J$ is exactly $-0.02$ (next cell).

**Solution**

In [ ]:
def matrix_J(C, G, a=-0.02, f=0.05):
    """The 2N x 2N matrix J of eq. 3.17 (the linearised model), for u = (x_1..x_N, y_1..y_N)."""
    N = len(C)
    s = C.sum(axis=1)                         # strength of each region: the sum of its fibres
    A = np.diag(a - G * s) + G * C
    B = 2 * np.pi * f * np.eye(N)             # B = diag(omega)
    return np.block([[A, -B], [B, A]])

In [ ]:
J = matrix_J(C, 8.0)
eigenvalues = np.linalg.eigvals(J)
print(f"J has shape {J.shape}; largest real part of its eigenvalues: {eigenvalues.real.max():.4f}")
print(f"so the slowest fluctuation is damped at {-eigenvalues.real.max():.4f} per second")

Each eigenvalue of $J$ belongs to one pattern of fluctuation across the regions, and its real part, with the sign
changed, is how fast that pattern is damped. The slowest one is damped at 0.02 per second, by $|a|$ alone. It is
the fluctuation shared by all regions, the **global signal**: the coupling acts only on differences, so it does
not damp it. It lasts longest and adds to every correlation. Every other pattern is damped more.

## Exercise 6: FC without simulation

Solve the Lyapunov equation 3.19 with `solve_continuous_lyapunov(J, M)` from scipy, which returns the $K$ that
solves $J K + K J^T = M$ (so here $M = -Q$). The last lines of the function, already written, turn the covariances
of the $x$ into correlations.

*Expected:* at $G = 8$ the FC of the linear theory correlates at about 0.94 with the FC simulated in Exercise 4,
which has a random error, and its fit to the data is about 0.66. The next cell then scans 40 values of $G$ in
about a second: best fit 0.665 at $G \approx 16$, and the mean FC equals that of the data at $G = 7.7$.

**Solution**

In [ ]:
def linear_fc(C, G, a=-0.02, f=0.05, sigma=0.02):
    """FC of the linear model, from the covariances K that solve the Lyapunov equation (eq. 3.19)."""
    N = len(C)
    J = matrix_J(C, G, a, f)
    K = solve_continuous_lyapunov(J, -sigma**2 * np.eye(2 * N))
    Kxx = K[:N, :N]                           # covariances of the x
    d = np.sqrt(np.diag(Kxx))                 # standard deviations of the x
    return Kxx / np.outer(d, d)

In [ ]:
FC_lin = linear_fc(C, 8.0)                          # used again in Part 6
FC_sim8 = np.corrcoef(X[list(Gs).index(8)].T)       # the simulation at G = 8 from Exercise 4
print(f"correlation between the linear FC and the simulated FC (G = 8): {fc_fit(FC_lin, FC_sim8):.3f}")
print(f"fit of the linear FC to the data: {fc_fit(FC_lin, FC_emp):.3f}")

G_grid = np.linspace(0.5, 30, 40)
fit_lin = [fc_fit(linear_fc(C, G), FC_emp) for G in G_grid]
mean_lin = [upper(linear_fc(C, G)).mean() for G in G_grid]

done_4 = len(fit) == len(Gs)                        # the points of Exercise 4, if it was completed
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(G_grid, fit_lin, label="linear theory")
if done_4:
    axes[0].plot(Gs, fit, "o", label="simulation")
axes[0].set_xlabel("G")
axes[0].set_ylabel("fit")
axes[0].legend()
axes[1].plot(G_grid, mean_lin, label="linear theory")
if done_4:
    axes[1].plot(Gs, mean_fc, "o", label="simulation")
axes[1].axhline(upper(FC_emp).mean(), color="#1A1A1A", ls=":", lw=1, label="data")
axes[1].set_xlabel("G")
axes[1].set_ylabel("mean FC")
axes[1].legend()
plt.tight_layout()
plt.show()

G_level = np.interp(upper(FC_emp).mean(), mean_lin, G_grid)
print(f"best fit {max(fit_lin):.3f} at G = {G_grid[np.argmax(fit_lin)]:.1f}; "
      f"the mean FC matches the data at G = {G_level:.1f}")

In the fit the simulations lie below the theory at every $G$, most at small $G$. Two things separate them from
the theory: the random error of a 20-minute run, and the differences between the two models, the cubic term
and the time step of the simulation (0.1 s). The random error is large. With 600 images each simulated
correlation has an error of about 0.07 at $G = 1$, as large as the spread of the true correlations; at $G = 8$
the error is 0.05 against a spread of 0.12. The data are a group average, with a much smaller error, so the
error of the model is not shared by the data and lowers the fit on average. The next cell separates the two
effects at $G = 8$: it simulates twenty runs with and without the cubic term (`cubic=False`), with the same
noise, and also averages their FC.

In [ ]:
# twenty runs of 20 minutes at G = 8, with and without the cubic term (a few seconds)
for cubic in (True, False):
    runs = simulate_hopf(C, np.full(20, 8.0), seed=3, cubic=cubic)
    FCs = [np.corrcoef(x.T) for x in runs]
    one_run = [fc_fit(F, FC_emp) for F in FCs]
    mean_one = [upper(F).mean() for F in FCs]
    FC_avg = np.mean(FCs, axis=0)
    print(f"cubic term {'kept:   ' if cubic else 'dropped:'} one run, fit {np.mean(one_run):.2f} +- "
          f"{np.std(one_run):.2f}, mean FC {np.mean(mean_one):.3f} +- {np.std(mean_one):.3f}; "
          f"FC averaged over 20 runs, fit {fc_fit(FC_avg, FC_emp):.2f}")
print(f"linear theory:       fit {fc_fit(FC_lin, FC_emp):.2f}, mean FC {upper(FC_lin).mean():.3f}")

With or without the cubic term one run fits at about 0.62, and the FC averaged over twenty runs at 0.66, as the
theory: most of the gap in the fit is random error. This is why the book averages the model over as many
simulations as there are participants (p. 47). The cubic term instead lowers the mean FC, from 0.290 to 0.272
here, while the fit hardly changes; the time step of the simulation lowers it from the 0.298 of the theory to
0.290. The mean FC of single runs also scatters, with a standard deviation of about 0.03:
the slowest pattern decays in $1/|a| = 50$ s, so 20 minutes hold few independent samples of it. A point above
the theory, as at $G = 20$ in Exercise 4, is this random error.

**Question 3.** The fit has a broad plateau: many values of $G$ reproduce the pattern of the FC almost equally
well. What else could fix $G$? And if the model is linear, why does it reproduce the FC with a fit of about 0.65?

**Question 4.** One parameter, $G$, scales all connections together. How would you change the model to fit the FC
better? What would you need to measure to recover the *direction* of the connections, which diffusion MRI cannot
see?

**Answer.** (Q3) The mean FC (here it gives $G \approx 8$), and the dynamics: how the FC changes in time (functional
connectivity dynamics, FCD, book ch. 2) and correlations with a delay (next lesson). With
the same $a$ and $\omega$ in every region, the covariances of the $x$ have a closed form,
$K_{xx} = \frac{\sigma^2}{2}\,(|a|\,\mathbf{1} + G L)^{-1}$, with $\mathbf{1}$ the identity matrix and
$L = \mathrm{diag}(s) - C$ the Laplacian of the SC (check it with `np.linalg.inv`). Writing
$|a|\,\mathbf{1} + G L = D - G C$ with $D = \mathrm{diag}(|a| + G s)$, the series
$K_{xx} = \frac{\sigma^2}{2} \sum_n (D^{-1} G C)^n D^{-1}$ converges (spectral radius 0.98) and adds up every path
between two regions, direct or indirect, so pairs with no direct fibre are still correlated. They are 69% of all pairs, and on them alone the
linear model at $G = 8$ still has a fit of 0.59, while the SC, zero on these pairs, predicts nothing there. The
FC depends neither on $\omega$ nor on $\sigma$, and on
$G$ and $a$ only through $G/|a|$, coupling against damping (for the linear model, with the same $a$ and $\omega$
everywhere).

**Answer.** (Q4) Give each connection its own effective weight and fit all of them, starting from the SC: this is the
generative effective connectivity (GEC, eq. 3.20). With FC alone the fitted matrix stays symmetric. In the data,
covariances with a delay, $\langle x_i(t + \tau)\, x_j(t) \rangle$, are not symmetric and reveal the direction
of the interactions; in this model, with symmetric $C$ and equal $\omega$, they are symmetric. This is the
starting point of the next lesson. The largest misfit of the model is between mirror
regions of the two hemispheres (homotopic pairs), the diagonals at $\pm 34$ in the FC matrix: their FC is 0.57 in
the data and 0.33 in the model, and for 12 of the 34 pairs tractography finds no fibre. The GEC of the book also
updates these connections (p. 46).

---
# Part 6 — Information in the model

Lesson 0 measured dependence in bits. For two jointly Gaussian signals the mutual information depends on their
correlation alone. For regions $i$ and $j$, whose correlation is $\mathrm{FC}_{ij}$,

$$\mathrm{MI}_{ij} = -\tfrac12 \log_2\big(1 - \mathrm{FC}_{ij}^2\big).$$

Lesson 0 wrote $\rho$ for this correlation and $I$ for the MI; here $\rho$ is the amplitude of a Hopf node (Part 3)
and $I$ a current or the inhibitory population, so we write FC and MI.

Lesson 0 defined the O-information of three signals as what two of them tell about the third separately, minus
what they tell together: $\Omega = I(Z;X) + I(Z;Y) - I(Z;X,Y)$. The last term splits in two steps, first $Y$,
then $X$:

$$I(Z;X,Y) = H(Z) - H(Z|X,Y) = \big[H(Z) - H(Z|Y)\big] + \big[H(Z|Y) - H(Z|X,Y)\big] = I(Z;Y) + I(Z;X|Y).$$

$I(Z;X|Y) = H(Z|Y) - H(Z|X,Y)$, the **conditional mutual information**, is the uncertainty on $Z$ that $X$
still removes once $Y$ is known. So $\Omega = I(Z;X) - I(Z;X|Y)$, which for regions $i$, $j$, $k$ we write
$\Omega = \mathrm{MI}_{ij} - \mathrm{MI}_{ij|k}$: the dependence of $i$ and $j$ minus what is left of it once
$k$ is known. $\Omega > 0$ when redundancy dominates, the same information shared by several signals; $\Omega < 0$
when synergy dominates, the information being only in the joint state, as in the triplet 26, 129, 142 of
Lesson 0, where region 129 behaves like the sum of two independent signals. A triplet with $\Omega < 0$ is called synergistic. $\Omega$ is a
balance: $\Omega \ge 0$ means that redundancy dominates, not that there is no synergy at all.

Near the working point the model is linear and driven by Gaussian noise (Part 5), so its fluctuations are jointly
Gaussian. At the same instant (zero lag) they are fixed by their covariances, and MI and $\Omega$ do not change when
a signal is rescaled: the FC of the linear theory gives every zero-lag quantity of Lesson 0. In particular the
$\Omega$ of a triplet is a function of its three correlations. If the model reproduced the FC exactly, it would
reproduce every Gaussian $\Omega$ too. The last lab therefore asks how the errors of the model on the pairs show up
in the triplets, through this nonlinear function of three entries of the FC; it is not a test of information
beyond the covariance.

For the data we do as Lesson 0 did with its triplets, and assume that the signals are jointly Gaussian, so that
the measured FC gives their $\Omega$. This FC is a group average over participants, with the negative
correlations set to zero (9 pairs are exactly 0). Lesson 0 used one recording of 200 regions, with many negative
correlations.

## Exercise 7: Mutual information in the model

Write `gaussian_mi(fc)`, the function `gaussian_mi` of Lesson 0, here with the argument `fc`. The cell then computes the MI of every pair of regions for
the linear model at $G = 8$ (`FC_lin`, from Exercise 6) and for the data.

*Expected:* a mean MI of 0.082 bits in the model and 0.092 bits in the data. The two patterns of MI correlate at
0.56, less than the fit of the FC (0.66): the MI is convex in |FC|, so the strongest pairs weigh most, and those are
the mirror pairs the model misses (without the 34 mirror pairs the correlation is 0.62). The mean FC is nearly the
same (0.298 and 0.291), but the correlations of the data are more spread (standard deviation 0.16 against 0.12).

**Solution**

In [ ]:
def gaussian_mi(fc):
    """Mutual information (bits) of two jointly Gaussian signals whose correlation is fc."""
    return -0.5 * np.log2(1 - np.asarray(fc) ** 2)


MI_model = gaussian_mi(upper(FC_lin))
MI_data = gaussian_mi(upper(FC_emp))
print(f"mean MI: model {MI_model.mean():.3f} bits, data {MI_data.mean():.3f} bits")
print(f"correlation between the two patterns of MI: {np.corrcoef(MI_model, MI_data)[0, 1]:.2f}")

## Demonstration: redundancy and synergy in 50,116 triplets

Lesson 0 computed $I(Z;X,Y) = -\tfrac12\log_2(1 - R^2)$, with
$R^2 = (\rho_{xz}^2 + \rho_{yz}^2 - 2\rho_{xy}\rho_{xz}\rho_{yz})/(1 - \rho_{xy}^2)$. Then
$1 - R^2 = \det F/(1 - \rho_{xy}^2)$, where $F$ is the $3 \times 3$ correlation matrix of the three signals, and
$\Omega = I(Z;X) + I(Z;Y) - I(Z;X,Y)$ becomes, for three regions $i$, $j$, $k$,

$$\Omega = \tfrac12\Big[\log_2 \det F - \log_2\big(1 - \mathrm{FC}_{ij}^2\big) - \log_2\big(1 - \mathrm{FC}_{ik}^2\big)
  - \log_2\big(1 - \mathrm{FC}_{jk}^2\big)\Big],$$

$$\det F = 1 - \mathrm{FC}_{ij}^2 - \mathrm{FC}_{ik}^2 - \mathrm{FC}_{jk}^2 + 2\,\mathrm{FC}_{ij}\,\mathrm{FC}_{ik}\,\mathrm{FC}_{jk}.$$

The next cell checks it on one triplet against `o_information_gauss`, a second formula for Gaussian signals:
$\Omega = TC - DTC$ (Optional B of Lesson 0), from $\det F$ and the diagonal of $F^{-1}$. Then it applies it to
all the triplets of the 68 regions, for the model and for the data. It also counts the triplets with a *missing link*: a pair
less correlated than the product of its correlations with the third region, such as
$\mathrm{FC}_{ij} < \mathrm{FC}_{ik}\,\mathrm{FC}_{jk}$. The triplet 26, 129, 142 of Lesson 0 is of this kind:
regions 26 and 142 do not correlate, yet both correlate with region 129.

In [ ]:
from itertools import combinations


def o_information_gauss(F):
    """O-information (bits) TC - DTC of jointly Gaussian variables with correlation matrix F."""
    F = np.asarray(F, dtype=float)
    log_det = np.log2(np.linalg.det(F))
    tc = -0.5 * log_det
    dtc = 0.5 * (log_det + np.sum(np.log2(np.diag(np.linalg.inv(F)))))
    return tc - dtc


def omega_triplets(FC, triplets):
    """O-information (bits) of every triplet (i, j, k) of regions, from the correlations FC."""
    i, j, k = triplets.T
    fc_ij, fc_ik, fc_jk = FC[i, j], FC[i, k], FC[j, k]
    det_F = 1 - fc_ij**2 - fc_ik**2 - fc_jk**2 + 2 * fc_ij * fc_ik * fc_jk
    return 0.5 * (np.log2(det_F) - np.log2(1 - fc_ij**2) - np.log2(1 - fc_ik**2) - np.log2(1 - fc_jk**2))


def missing_link(FC, triplets):
    """True for each triplet with a pair less correlated than the product of its correlations with the third."""
    i, j, k = triplets.T
    fc_ij, fc_ik, fc_jk = FC[i, j], FC[i, k], FC[j, k]
    return (fc_ij < fc_ik * fc_jk) | (fc_ik < fc_ij * fc_jk) | (fc_jk < fc_ij * fc_ik)


triplets = np.array(list(combinations(range(N), 3)))
omega_model = omega_triplets(FC_lin, triplets)
omega_data = omega_triplets(FC_emp, triplets)
first = triplets[0]
print(f"check on one triplet: {omega_model[0]:.4f} bits; o_information_gauss gives "
      f"{o_information_gauss(FC_lin[np.ix_(first, first)]):.4f} bits")
print(f"{len(triplets)} triplets; synergistic (Omega < 0): model {np.sum(omega_model < 0)}, "
      f"data {np.sum(omega_data < 0)} ({np.mean(omega_data < 0):.0%})")
print(f"Omega, model: from {omega_model.min():.3f} to {omega_model.max():.3f} bits; "
      f"data: from {omega_data.min():.3f} to {omega_data.max():.3f} bits")
print(f"correlation of Omega between model and data, over all triplets: "
      f"{np.corrcoef(omega_model, omega_data)[0, 1]:.2f}")
link_model, link_data = missing_link(FC_lin, triplets), missing_link(FC_emp, triplets)
print(f"triplets with a missing link: model {link_model.sum()}, data {link_data.sum()}; "
      f"synergistic triplets of the data with a missing link: {np.sum(link_data & (omega_data < 0))}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
bins = np.linspace(-0.08, 0.30, 77)
axes[0].axvspan(-0.08, 0, color="#EDEDED", zorder=0)
axes[0].hist(omega_data, bins=bins, alpha=0.75, label="data")
axes[0].hist(omega_model, bins=bins, alpha=0.75, label="model (G = 8)")
axes[0].axvline(0, color="#1A1A1A", lw=0.8)
axes[0].set_xlim(-0.08, 0.30)
top = axes[0].get_ylim()[1]
axes[0].text(-0.04, 0.9 * top, "synergy", ha="center", fontsize=9, color="#555555")
axes[0].text(0.15, 0.9 * top, "redundancy", ha="center", fontsize=9, color="#555555")
axes[0].set_xlabel("Ω of a triplet (bits)")
axes[0].set_ylabel("number of triplets")
axes[0].legend()
axes[1].plot(omega_data, omega_model, ".", ms=1, alpha=0.15)
axes[1].axvline(0, color="#1A1A1A", lw=0.8)
axes[1].axhline(0, color="#1A1A1A", lw=0.8)
axes[1].set_xlabel("Ω, data (bits)")
axes[1].set_ylabel("Ω, model (bits)")
plt.tight_layout()
plt.show()

Left: how many triplets have each value of $\Omega$, up to 0.3 bits (a few triplets of the data reach 0.62); the
grey band is synergy, $\Omega < 0$. Right: each point is a triplet, data against model.

**Question 5.** Every triplet of the model is redundancy-dominated, while 3% of the triplets in the data are
synergistic. Why can this model not produce a synergistic triplet (look at the missing links)? Which of its
assumptions would you change?

**Answer.** The model of this notebook has the same $\omega$ and the same noise in every region, and symmetric couplings
$G C_{ij} \ge 0$. Then the inverse covariance of the $x$ is proportional to $|a|\,\mathbf{1} + G L$ (or to
$\mathrm{diag}(|a_j|) + G L$ if $a$ differs between regions): all its off-diagonal entries, $-G C_{ij}$, are
$\le 0$. Every triplet inherits this sign (its inverse covariance is a Schur complement, which keeps it), so all
three partial correlations of a triplet are $\ge 0$: $\mathrm{FC}_{ij} \ge \mathrm{FC}_{ik}\,\mathrm{FC}_{jk}$
for all three pairs. For three Gaussian signals this implies $\mathrm{MI}_{ij|k} \le \mathrm{MI}_{ij}$, so
$\Omega \ge 0$. Proof: write the inverse covariance of the triplet with diagonal $p_i$ and off-diagonal entries
$-q_{ij} \le 0$. The partial correlation of $i$ and $j$ is $q_{ij}/\sqrt{p_i p_j}$, and their correlation is
$(q_{ij}\,p_k + q_{ik}\,q_{jk})/\sqrt{(p_i p_k - q_{ik}^2)(p_j p_k - q_{jk}^2)}$, which is at least
$q_{ij}\,p_k/(p_k\sqrt{p_i p_j})$, the partial correlation $r$; and for Gaussian signals
$\mathrm{MI}_{ij|k} = -\tfrac12\log_2(1 - r^2)$. So the model has no missing link and no synergistic
triplet.

The result holds for this homogeneous version of the model, not for every linear Gaussian model with positive
couplings. With different frequencies in the regions the inverse covariance can have positive entries: three
regions with couplings $G C$ of 0.1 between the first and each of the others and frequencies 0.05, 0.04 and
0.07 Hz give $\Omega = -0.02$ bits (Optional A). Negative or asymmetric couplings, different frequencies or other
changes of the dynamics can remove the constraint; a different $a$ in each region alone cannot. (With the SC and
the frequencies of Optional A, the 68 regions at $G = 8$ still have no synergistic triplet.)

Since all the FC of the data are $\ge 0$, synergy there requires a missing link, a negative partial correlation,
as in the triplet 26, 129, 142 of Lesson 0; 4830 triplets have one and 1580 of them are synergistic. Why only 3% in the data,
against about half in Lesson 0? Lesson 0 used one recording with many negative correlations, and a triplet with
an odd number of them always has $\Omega < 0$. Here the FC is a group average with the negative values set to
zero, and 36% of its synergistic triplets contain one of the 9 pairs set exactly to 0. Moreover the $\Omega$ of
an average FC is not the average of the $\Omega$ of the participants, because $\Omega$ is a nonlinear function of
the correlations. The two percentages describe two different estimates, and neither measures how common synergy
is in the brain. The redundancy itself is partly reproduced (correlation 0.59 over all triplets), much as the FC
is.

---
# Optional

**A. A frequency for each region.** In the book the frequencies $\omega_j$ are estimated region by region from the
data. Pass an array of 68 frequencies to `linear_fc`, for example
`f = np.random.default_rng(0).uniform(0.04, 0.07, N)`. Does the fit change? Then take three regions, the first
coupled to the other two, `C3 = np.array([[0, 0.1, 0.1], [0.1, 0, 0], [0.1, 0, 0]])`, compute
`F3 = linear_fc(C3, 1.0, f=np.array([0.05, 0.04, 0.07]))` and the $\Omega$ of the triplet with
`o_information_gauss(F3)`. All couplings are positive and symmetric: what is the sign of $\Omega$, and why
does Question 5 not forbid it?

**B. From firing rates to the BOLD signal (eq. 3.7–3.11).** In the DMF the BOLD signal is obtained from the rate
of the E population through a model of the blood flow, the Balloon–Windkessel model of Stephan et al. (2007):
active neurons make the blood vessels widen, more oxygenated blood arrives than is used, and the BOLD signal
rises. The next cell computes the response to 1 s at 5 Hz instead of 3 Hz. The letters of eq. 3.7–3.11 are those
of the book and have nothing to do with the same letters elsewhere in this notebook. The code makes two changes to
the printed equations. The input of eq. 3.7 is centred on the baseline rate, $0.5\,(r - 3)$: with $0.5\,r + 3$ the
blood flow at rest would settle at about 12 times its baseline. Eq. 3.10 uses the standard term
$f\,[1 - (1 - E_0)^{1/f}]/E_0$ of Stephan et al. (2007), where the book drops the "1 −"; here $f$ is the blood
inflow and $E_0 = 0.34$ the fraction of oxygen extracted at rest ($\rho$ in the book).

In [ ]:
def bold(r_E, dt, r_base=3.0, k=0.65, gamma=0.41, tau=0.98, alpha=0.32, E0=0.34, V0=0.02):
    """BOLD signal (eq. 3.7-3.11) from the rate r_E (Hz) of the E population, sampled every dt seconds."""
    k1, k2, k3 = 7 * E0, 2.0, 2 * E0 - 0.2
    s, f, v, q = 0.0, 1.0, 1.0, 1.0      # vasodilatory signal, blood inflow, volume, deoxyhaemoglobin
    B = np.empty(len(r_E))
    for i, r in enumerate(r_E):
        s += dt * (0.5 * (r - r_base) - k * s - gamma * (f - 1))
        f += dt * s
        v += dt * (f - v ** (1 / alpha)) / tau
        q += dt * (f * (1 - (1 - E0) ** (1 / f)) / E0 - q * v ** (1 / alpha) / v) / tau
        B[i] = V0 * (k1 * (1 - q) + k2 * (1 - q / v) + k3 * (1 - v))
    return B


dt = 1e-3
t = np.arange(0, 30, dt)
r = np.where((t >= 5) & (t < 6), 5.0, 3.0)         # 1 s at 5 Hz instead of 3 Hz
plt.figure(figsize=(6, 3))
plt.plot(t, 100 * bold(r, dt))
plt.axvspan(5, 6, color="#E6E6E6")
plt.xlabel("time (s)")
plt.ylabel("BOLD (% signal change)")
plt.show()

---
# Summary

- Noise turns the hard threshold of a spiking neuron into a smooth rate curve. The DMF describes each region by two
  populations (E and I) with such a curve and their open receptors. On its own a region fires steadily at about
  3 Hz, with no rhythm, and FIC keeps every region of the network at 3 Hz.
- The Hopf node is the normal form of an oscillation near its onset. Just below the bifurcation ($a = -0.02$), noise
  drives irregular oscillations with a mean amplitude of about two thirds of the radius $\sqrt{a}$ of the limit cycle.
- Coupled through the SC with one global parameter $G$, the Hopf model reproduces the measured FC with a fit of
  about 0.65. Near the working point the model is linear, so the FC follows from a Lyapunov equation, with no
  simulation.
- For jointly Gaussian signals the FC gives the mutual information of every pair and the O-information of every
  triplet (Lesson 0). With equal frequencies and symmetric positive couplings the model has no synergistic
  triplet; through the FC it reproduces the pairs and part of the redundancy of the triplets.
- The FC alone hardly fixes $G$, and one parameter cannot describe connections with a direction. The next lesson
  starts from connections with a direction, fitted to correlations with a delay, and reviews what these models
  have taught us.

**Reading:** book ch. 1 and ch. 3 (pp. 1–8 and 37–47).

**References**
- Abbott LF, Chance FS (2005). Drivers and modulators from push-pull and balanced synaptic input. *Prog Brain Res* 149:147–155.
- Cover TM, Thomas JA (2006). *Elements of Information Theory*, 2nd ed. Wiley. Chapters 2 and 8.
- Deco G, Ponce-Alvarez A, Mantini D, Romani GL, Hagmann P, Corbetta M (2013). Resting-state functional connectivity emerges from structurally and dynamically shaped slow linear fluctuations. *J Neurosci* 33:11239–11252.
- Deco G, Ponce-Alvarez A, Hagmann P, Romani GL, Mantini D, Corbetta M (2014). How local excitation–inhibition ratio impacts the whole brain dynamics. *J Neurosci* 34:7886–7898.
- Deco G, Kringelbach ML, Jirsa VK, Ritter P (2017). The dynamics of resting fluctuations in the brain: metastability and its dynamical cortical core. *Sci Rep* 7:3095.
- Larivière S et al. (2021). The ENIGMA Toolbox: multiscale neural contextualization of multisite neuroimaging datasets. *Nat Methods* 18:698–700.
- Rosas FE, Mediano PAM, Gastpar M, Jensen HJ (2019). Quantifying high-order interdependencies via multivariate extensions of the mutual information. *Phys Rev E* 100:032305.
- Stephan KE, Weiskopf N, Drysdale PM, Robinson PA, Friston KJ (2007). Comparing hemodynamic models with DCM. *NeuroImage* 38:387–401.
- Strogatz SH (2015). *Nonlinear Dynamics and Chaos*, 2nd ed. Westview Press. Section 8.2 for the Hopf bifurcation.
- Wong KF, Wang XJ (2006). A recurrent network mechanism of time integration in perceptual decisions. *J Neurosci* 26:1314–1328.